# Lab 5: Weights & Biases Experiment Tracking and Artifact Management

**Course:** ML Operations  
**Objective:** Set up W&B for experiment tracking, log parameters, metrics, and model artifacts, and manage multiple experiment runs.

## 5.1 Setup and Imports

In [1]:
import os
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(repo_root)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

print(f'Working directory: {Path.cwd()}')

Working directory: /home/d1pg1/ml_engineering_labs


In [2]:
import yaml
import pandas as pd
import wandb
import torch
import torch.nn as nn

from src.data.dataset import process_data, create_data_loader, train_transform, eval_transform
from src.models.cnn import CifarCNN
from src.training.wandb_trainer import train_with_wandb
from src.training.evaluate import test_model

print(f'wandb version: {wandb.__version__}')
print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

wandb version: 0.27.0
PyTorch version: 2.12.0+cu130
Device: cpu


## 5.2 Part 1: Weights & Biases Setup

### 5.2.1 Authentication

W&B requires a free account at https://wandb.ai. After account creation, retrieve your API key from
https://wandb.ai/settings and set it as an environment variable before running this notebook:

```bash
export WANDB_API_KEY=<your-key>
```

The cell below reads the key from the environment and calls `wandb.login()`. If the variable is not
set, W&B will fall back to an interactive browser login prompt.

In [3]:
api_key = os.environ.get('WANDB_API_KEY')
if api_key:
    wandb.login(key=api_key)
    print('Logged in via WANDB_API_KEY environment variable.')
else:
    wandb.login()
    print('Logged in interactively.')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.


wandb: Appending key for api.wandb.ai to your netrc file: /home/d1pg1/.netrc


wandb: Currently logged in as: d1pg1 (d1pg1-national-technical-university-kharkiv-polytechnic-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in via WANDB_API_KEY environment variable.


## 5.3 Part 2: Logging Parameters, Configs, and Metrics

### 5.3.1 Load configuration

Base config from `configs/base.yaml` provides data splits and default training hyperparameters.
Lab-specific overrides live in `configs/lab05.yaml` (project name, run names, sweep LRs).

In [4]:
with open('configs/base.yaml') as f:
    base_cfg = yaml.safe_load(f)

with open('configs/lab05.yaml') as f:
    lab05_cfg = yaml.safe_load(f)

main_cfg = lab05_cfg['main_run']
PROJECT_NAME = lab05_cfg['project_name']

print('Base config:', base_cfg)
print('Lab05 main run config:', main_cfg)
print('W&B project:', PROJECT_NAME)

Base config: {'data': {'test_size': 0.2, 'val_size': 0.2, 'random_state': 42, 'cifar_url': 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', 'data_dir': 'data', 'n_classes': 10}, 'training': {'lr': 0.001, 'batch_size': 128, 'num_epochs': 20, 'num_workers': 2}, 'output': {'save_path': 'outputs/best_model.pth'}}
Lab05 main run config: {'num_epochs': 10, 'lr': 0.001, 'batch_size': 128, 'run_name': 'Run 1 - Default Config'}
W&B project: mlops-cifar10


### 5.3.2 Prepare datasets (reuse existing extracted CIFAR-10 images)

In [5]:
LABELS_PATH = 'data/cifar-10-batches-py'
IMAGES_DIR  = 'data/images'

train_df, val_df, test_df = process_data(
    images_dir=IMAGES_DIR,
    labels_path=LABELS_PATH,
    config=base_cfg['data'],
)

loader_cfg_train = {
    'batch_size': main_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': train_transform,
}
loader_cfg_eval = {
    'batch_size': main_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': eval_transform,
}

train_loader = create_data_loader(IMAGES_DIR, train_df, loader_cfg_train)
val_loader   = create_data_loader(IMAGES_DIR, val_df,   loader_cfg_eval)
test_loader  = create_data_loader(IMAGES_DIR, test_df,  loader_cfg_eval)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

2026-05-14 13:47:19,580 INFO Loading batch: data_batch_1


/home/d1pg1/ml_engineering_labs/src/data/dataset.py:90: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  batch = pickle.load(f, encoding="bytes")


2026-05-14 13:47:20,251 INFO Loading batch: data_batch_2


2026-05-14 13:47:20,829 INFO Loading batch: data_batch_3


2026-05-14 13:47:21,509 INFO Loading batch: data_batch_4


2026-05-14 13:47:21,938 INFO Loading batch: data_batch_5


2026-05-14 13:47:22,365 INFO Loading batch: test_batch


2026-05-14 13:47:22,789 INFO Loaded 60000 images from CIFAR-10 batches.


2026-05-14 13:47:22,830 INFO Prepared 3 data splits — train: 38400, val: 9600, test: 12000


Train batches: 300 | Val batches: 75 | Test batches: 94


### 5.3.3 Main training run — log params and per-epoch metrics via W&B

Parameters are passed atomically as a `config=` dict to `wandb.init()` — they appear in the
run's **Config** panel immediately. Per-epoch losses are streamed via `wandb.log()` inside
`train_with_wandb()`. Final test metrics are logged after evaluation.

In [6]:
MAIN_SAVE_PATH = Path('outputs/lab05_main.pth')

model     = CifarCNN(n_classes=base_cfg['data']['n_classes'])
optimizer = torch.optim.Adam(model.parameters(), lr=main_cfg['lr'])
loss_fn   = nn.CrossEntropyLoss()

wandb_config = {
    'lr':           main_cfg['lr'],
    'batch_size':   main_cfg['batch_size'],
    'num_epochs':   main_cfg['num_epochs'],
    'optimizer':    'Adam',
    'test_size':    base_cfg['data']['test_size'],
    'val_size':     base_cfg['data']['val_size'],
    'random_state': base_cfg['data']['random_state'],
    'n_classes':    base_cfg['data']['n_classes'],
    'model':        'CifarCNN',
}

run = wandb.init(
    project=PROJECT_NAME,
    name=main_cfg['run_name'],
    config=wandb_config,
)
print(f'W&B Run ID:  {run.id}')
print(f'W&B Run URL: {run.url}')

best_path = train_with_wandb(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=loss_fn,
    optimizer=optimizer,
    num_epochs=main_cfg['num_epochs'],
    device=device,
    save_path=MAIN_SAVE_PATH,
)

test_metrics = test_model(model, test_loader, loss_fn, device)
wandb.log({
    'test_loss':      test_metrics['loss'],
    'test_accuracy':  test_metrics['accuracy'],
    'test_precision': test_metrics['precision'],
    'test_recall':    test_metrics['recall'],
    'test_f1':        test_metrics['f1'],
})

print('\n=== Main Run Results ===')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

wandb.finish()

wandb: setting up run cbb52r1w


wandb: Tracking run with wandb version 0.27.0


wandb: Run data is saved locally in /home/d1pg1/ml_engineering_labs/wandb/run-20260514_134723-cbb52r1w
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run Run 1 - Default Config


wandb: ⭐️ View project at https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10


...


=== Main Run Results ===
  loss: 0.8248
  accuracy: 0.7112
  precision: 0.7176
  recall: 0.7112
  f1: 0.7030


wandb: uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:          epoch ▁▂▃▃▄▅▆▆▇█
wandb:  test_accuracy ▁
wandb:        test_f1 ▁
wandb:      test_loss ▁
wandb: test_precision ▁
wandb:    test_recall ▁
wandb:     train_loss █▅▃▃▂▂▂▁▁▁
wandb:       val_loss █▆▄▃▂▂▁▁▁▁
wandb: 
wandb: Run summary:
wandb:          epoch 10
wandb:  test_accuracy 0.71117
wandb:        test_f1 0.703
wandb:      test_loss 0.82483
wandb: test_precision 0.7176
wandb:    test_recall 0.71117
wandb:     train_loss 0.85209
wandb:       val_loss 0.82551
wandb: 


wandb: 🚀 View run Run 1 - Default Config at: https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10/runs/cbb52r1w
wandb: ⭐️ View project at: https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260514_134723-cbb52r1w/logs


## 5.4 Part 3: Logging Output Artifacts

Log the best model checkpoint as a W&B Artifact of type `'model'`, then demonstrate
download and inference.

> **Note:** W&B artifacts must be logged inside an active run (before `wandb.finish()`). In
> production, log the artifact at the end of the training run itself. Here a separate run is
> opened for pedagogical clarity matching the lab structure.

In [7]:
run_artifact = wandb.init(
    project=PROJECT_NAME,
    name=main_cfg['run_name'] + ' (artifact logging)',
    job_type='artifact-logging',
)

artifact = wandb.Artifact(
    name='cifar10-cnn',
    type='model',
    description='Best CifarCNN checkpoint for lab05 main run',
    metadata=wandb_config,
)
artifact.add_file(str(MAIN_SAVE_PATH))
run_artifact.log_artifact(artifact)

print(f'Artifact logged: {MAIN_SAVE_PATH}')
run_artifact.finish()

wandb: setting up run e9ijh5v2


wandb: Tracking run with wandb version 0.27.0


wandb: Run data is saved locally in /home/d1pg1/ml_engineering_labs/wandb/run-20260514_140549-e9ijh5v2
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run Run 1 - Default Config (artifact logging)


wandb: ⭐️ View project at https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10


...

Artifact logged: outputs/lab05_main.pth


wandb: uploading artifact cifar10-cnn


wandb: uploading data


wandb: 🚀 View run Run 1 - Default Config (artifact logging) at: https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10/runs/e9ijh5v2
wandb: ⭐️ View project at: https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260514_140549-e9ijh5v2/logs


In [8]:
# Demonstrate artifact retrieval and inference
api = wandb.Api()
artifact_path = f'{run_artifact.entity}/{PROJECT_NAME}/cifar10-cnn:latest'
downloaded_artifact = api.artifact(artifact_path)
local_dir = downloaded_artifact.download()
local_path = Path(local_dir) / 'lab05_main.pth'
print(f'Downloaded to: {local_path}')

loaded_model = CifarCNN(n_classes=base_cfg['data']['n_classes'])
loaded_model.load_state_dict(torch.load(local_path, map_location=device))
loaded_model.eval()

from src.data.dataset import CIFAR10_CLASSES
sample_inputs, sample_labels = next(iter(test_loader))
sample_inputs = sample_inputs[:8].to(device)
with torch.no_grad():
    preds = torch.argmax(loaded_model(sample_inputs), dim=1).cpu()

print('\nArtifact inference demo (first 8 test samples):')
print(f'{"Predicted":<14} {"Ground truth"}')
for p, t in zip(preds.tolist(), sample_labels[:8].tolist()):
    match = 'OK' if p == t else 'WRONG'
    print(f'  {CIFAR10_CLASSES[p]:<12} {CIFAR10_CLASSES[t]} {match}')

wandb:   1 of 1 files downloaded.  


Downloaded to: /home/d1pg1/ml_engineering_labs/artifacts/cifar10-cnn:v0/lab05_main.pth



Artifact inference demo (first 8 test samples):
Predicted      Ground truth
  airplane     airplane OK
  deer         deer OK
  automobile   automobile OK
  deer         frog WRONG
  horse        horse OK
  bird         bird OK
  dog          dog OK
  automobile   ship WRONG


## 5.5 Part 4: Experimentation with W&B

Three sweep runs over `learning_rates: [0.01, 0.001, 0.0001]` with 5 epochs each.
Run names follow the structured naming convention: `"Run 2 - LR=0.01"`, `"Run 3 - LR=0.001"`, etc.
Each run gets its own `wandb.init()` / `wandb.finish()` pair and its model logged as an artifact.

In [9]:
sweep_cfg = lab05_cfg['sweep']
sweep_results = []

loader_cfg_sweep_train = {
    'batch_size': sweep_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': train_transform,
}
loader_cfg_sweep_eval = {
    'batch_size': sweep_cfg['batch_size'],
    'num_workers': base_cfg['training']['num_workers'],
    'transform': eval_transform,
}
sweep_train_loader = create_data_loader(IMAGES_DIR, train_df, loader_cfg_sweep_train)
sweep_val_loader   = create_data_loader(IMAGES_DIR, val_df,   loader_cfg_sweep_eval)
sweep_test_loader  = create_data_loader(IMAGES_DIR, test_df,  loader_cfg_sweep_eval)

for i, lr in enumerate(sweep_cfg['learning_rates'], start=2):
    run_name  = f'Run {i} - LR={lr}'
    save_path = Path(f'outputs/lab05_sweep_lr{str(lr).replace(".", "")}.pth')
    print(f'\n--- Starting run: {run_name} ---')

    sweep_model     = CifarCNN(n_classes=base_cfg['data']['n_classes'])
    sweep_optimizer = torch.optim.Adam(sweep_model.parameters(), lr=lr)
    sweep_loss_fn   = nn.CrossEntropyLoss()

    sweep_run = wandb.init(
        project=PROJECT_NAME,
        name=run_name,
        config={
            'lr':          lr,
            'batch_size':  sweep_cfg['batch_size'],
            'num_epochs':  sweep_cfg['num_epochs'],
            'optimizer':   'Adam',
            'n_classes':   base_cfg['data']['n_classes'],
            'model':       'CifarCNN',
        },
    )
    print(f'  W&B Run URL: {sweep_run.url}')

    train_with_wandb(
        model=sweep_model,
        train_loader=sweep_train_loader,
        val_loader=sweep_val_loader,
        loss_function=sweep_loss_fn,
        optimizer=sweep_optimizer,
        num_epochs=sweep_cfg['num_epochs'],
        device=device,
        save_path=save_path,
    )

    sweep_metrics = test_model(sweep_model, sweep_test_loader, sweep_loss_fn, device)
    wandb.log({
        'test_loss':     sweep_metrics['loss'],
        'test_accuracy': sweep_metrics['accuracy'],
        'test_precision': sweep_metrics['precision'],
        'test_recall':   sweep_metrics['recall'],
        'test_f1':       sweep_metrics['f1'],
    })

    sweep_artifact = wandb.Artifact(
        name=f'cifar10-cnn-sweep-lr{str(lr).replace(".", "")}',
        type='model',
    )
    sweep_artifact.add_file(str(save_path))
    sweep_run.log_artifact(sweep_artifact)

    sweep_results.append({
        'run_name':  run_name,
        'lr':        lr,
        'accuracy':  sweep_metrics['accuracy'],
        'f1':        sweep_metrics['f1'],
        'test_loss': sweep_metrics['loss'],
    })
    print(f'  Accuracy: {sweep_metrics["accuracy"]:.4f} | F1: {sweep_metrics["f1"]:.4f}')
    wandb.finish()


--- Starting run: Run 2 - LR=0.01 ---


wandb: setting up run dfjacvuz


wandb: Tracking run with wandb version 0.27.0


wandb: Run data is saved locally in /home/d1pg1/ml_engineering_labs/wandb/run-20260514_140559-dfjacvuz
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run Run 2 - LR=0.01


...

wandb: uploading artifact cifar10-cnn-sweep-lr00001


wandb: uploading history steps 5-5, summary


wandb: 
wandb: Run history:
wandb:          epoch ▁▃▅▆█
wandb:  test_accuracy ▁
wandb:        test_f1 ▁
wandb:      test_loss ▁
wandb: test_precision ▁
wandb:    test_recall ▁
wandb:     train_loss █▄▃▂▁
wandb:       val_loss █▅▃▂▁
wandb: 
wandb: Run summary:
wandb:          epoch 5
wandb:  test_accuracy 0.6545
wandb:        test_f1 0.65333
wandb:      test_loss 0.97848
wandb: test_precision 0.65796
wandb:    test_recall 0.6545
wandb:     train_loss 1.11267
wandb:       val_loss 0.97745
wandb: 


wandb: 🚀 View run Run 4 - LR=0.0001 at: https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10/runs/fjf1c233
wandb: ⭐️ View project at: https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260514_142458-fjf1c233/logs


In [10]:
results_df = pd.DataFrame(sweep_results)
print('\n=== Sweep Results Summary ===')
print(results_df.to_string(index=False))
print(f'\nView all runs: https://wandb.ai/{wandb.api.default_entity}/{PROJECT_NAME}')


=== Sweep Results Summary ===
         run_name     lr  accuracy       f1  test_loss
  Run 2 - LR=0.01 0.0100   0.32775 0.275619   1.656529
 Run 3 - LR=0.001 0.0010   0.68475 0.683876   0.889025
Run 4 - LR=0.0001 0.0001   0.65450 0.653334   0.978481



View all runs: https://wandb.ai/d1pg1-national-technical-university-kharkiv-polytechnic-/mlops-cifar10


## 5.6 Part 5: Lab Report

See [docs/lab_report_05.md](../docs/lab_report_05.md) for the full written report.